In [3]:
"""
================================================================================
                    临床特征综合统计分析系统(精简版)
================================================================================

功能概述：
    该脚本针对神经内分泌肿瘤(P-NEN)临床特征进行多层次统计分析

主要分析方法：
    1️⃣  数据预处理：缺失值分析、特征清洗、样本筛选
    2️⃣  单因素分析：Kruskal-Wallis非参数检验 + Spearman趋势相关性分析
    3️⃣  多因素分析：有序Logistic回归模型(Proportional Odds Model)

输出规范：
    📁 所有结果保存在 ./clinical/ 文件夹
    📊 统计表格：CSV格式，包含详细统计指标
    📈 关键图表：高分辨率PNG格式(300 DPI)，出版级质量

================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.linear_model import LogisticRegression
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# 设置matplotlib参数
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)

# 配色方案
COLORS = {
    'primary': '#2E86AB',
    'danger': '#D62246',
    'warning': '#F77F00',
    'info': '#4361EE',
    'success': '#06A77D',
}


def create_output_folder(base_dir='./clinical'):
    """创建输出文件夹"""
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
        print(f"📁 创建文件夹: {base_dir}")
    return base_dir


def load_and_encode_data():
    """加载数据并编码标签"""
    print("="*80)
    print("步骤1：数据加载与标签编码")
    print("="*80)
    
    clinical = pd.read_excel('./data/P-NEN.xlsx', index_col=0)
    
    label = clinical['Grade']
    y = label.map({'G1': 0, 'G2': 1, 'G3': 2, '癌': 2})
    
    features = ['size', 'sex', 'Age', 'Height(m)', 'SMI', 'IFI', 'SFI', 'VFI',
                'VF/SF', 'TFI', 'RIF%', 'RSF%', 'RVF%', 'CA19-9', 'CA125', 'CA72-4',
                'CEA', 'AFP', 'NLR', 'PLR', 'SII', 'LDH', 'SCr', 'eGFR', 'FBG',
                'TC', 'TG', 'ALT', 'AST', 'ALB', 'BUN']
    X = clinical[features].copy()
    Surv = clinical.loc[:, ['time', 'event']]
    
    valid_idx = y.notna()
    print(f"✓ 原始样本数: {len(y)}")
    print(f"✓ 标签有效样本数: {valid_idx.sum()}")
    
    if valid_idx.sum() < len(y):
        print(f"⚠ 删除标签缺失的 {(~valid_idx).sum()} 个样本")
        X = X[valid_idx]
        y = y[valid_idx]
    
    y = y.astype(int)
    print(f"\n标签分布: G1={sum(y==0)}, G2={sum(y==1)}, G3/癌={sum(y==2)}")
    print(f"总样本数: {len(y)}")
    
    return X, y, features, Surv


def analyze_missing_values(X, output_dir):
    """缺失值分析"""
    print("\n" + "="*80)
    print("步骤2：缺失值分析与处理")
    print("="*80)
    
    missing_rate = X.isnull().sum() / len(X) * 100
    
    print("\n各特征缺失率：")
    missing_df = pd.DataFrame({
        'Feature': missing_rate.index,
        'Missing_Rate(%)': missing_rate.values
    }).sort_values('Missing_Rate(%)', ascending=False)
    
    print(missing_df[missing_df['Missing_Rate(%)'] > 0].to_string(index=False))
    
    missing_df.to_csv(os.path.join(output_dir, '01_missing_rate_analysis.csv'),
                      index=False, encoding='utf-8-sig')
    
    # 绘制缺失率条形图
    if missing_df['Missing_Rate(%)'].max() > 0:
        missing_plot = missing_df[missing_df['Missing_Rate(%)'] > 0].sort_values('Missing_Rate(%)')
        
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = plt.cm.RdYlGn_r(missing_plot['Missing_Rate(%)'] / 100)
        
        bars = ax.barh(range(len(missing_plot)), missing_plot['Missing_Rate(%)'],
                       color=colors, edgecolor='white', linewidth=1.5, alpha=0.85)
        
        ax.axvline(x=25, color=COLORS['danger'], linestyle='--', linewidth=2.5,
                   label='25% Threshold', alpha=0.8)
        
        ax.set_yticks(range(len(missing_plot)))
        ax.set_yticklabels(missing_plot['Feature'], fontsize=10)
        ax.set_xlabel('Missing Rate (%)', fontsize=12, fontweight='bold')
        ax.set_title('Missing Value Analysis by Feature', fontsize=13, fontweight='bold')
        ax.legend(loc='lower right', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
        
        for i, (bar, rate) in enumerate(zip(bars, missing_plot['Missing_Rate(%)'])):
            ax.text(rate + 1, i, f'{rate:.1f}%', va='center', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '01_missing_rate_barplot.png'),
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"📊 保存图片: 01_missing_rate_barplot.png")
    
    cols_to_drop = missing_rate[missing_rate >= 25].index.tolist()
    cols_to_keep = missing_rate[missing_rate < 25].index.tolist()
    
    if cols_to_drop:
        print(f"\n⚠ 剔除缺失率≥25%的特征 ({len(cols_to_drop)}个): {cols_to_drop}")
        X = X[cols_to_keep]
    
    return X, cols_to_drop


def impute_missing_values(X):
    """插补缺失值"""
    print("\n" + "="*80)
    print("步骤3：缺失值插补")
    print("="*80)
    
    categorical_vars = ['sex', 'Metastasis']
    continuous_vars = [col for col in X.columns if col not in categorical_vars]
    
    numeric_vars = [col for col in continuous_vars if col in X.columns]
    if X[numeric_vars].isnull().sum().sum() > 0:
        print("使用MICE多重插补法处理连续变量缺失值...")
        imputer = IterativeImputer(random_state=42, max_iter=10, min_value=0)
        X[numeric_vars] = imputer.fit_transform(X[numeric_vars])
        print("✓ 连续变量插补完成")
    
    for col in categorical_vars:
        if col in X.columns and X[col].isnull().sum() > 0:
            X[col].fillna(X[col].mode()[0], inplace=True)
            print(f"✓ {col} 使用众数填充")
    
    return X, categorical_vars, continuous_vars


def encode_features(X, categorical_vars):
    """特征编码"""
    print("\n" + "="*80)
    print("步骤4：特征编码")
    print("="*80)
    
    cat_in_data = [col for col in categorical_vars if col in X.columns]
    X_encoded = pd.get_dummies(X, columns=cat_in_data, drop_first=True)
    
    print(f"✓ 编码前特征数: {X.shape[1]}")
    print(f"✓ 编码后特征数: {X_encoded.shape[1]}")
    
    return X_encoded


def univariate_analysis(X_encoded, y, output_dir):
    """单因素Kruskal-Wallis分析"""
    print("\n" + "="*80)
    print("步骤5：单因素分析（Kruskal-Wallis检验）")
    print("="*80)
    
    X_encoded = X_encoded.reset_index(drop=True)
    y = y.reset_index(drop=True)
    
    univariate_results = []
    
    for col in X_encoded.columns:
        try:
            group0 = X_encoded.loc[y == 0, col].dropna()
            group1 = X_encoded.loc[y == 1, col].dropna()
            group2 = X_encoded.loc[y == 2, col].dropna()
            
            if len(group0) >= 3 and len(group1) >= 3 and len(group2) >= 3:
                stat, p_value = stats.kruskal(group0, group1, group2)
                
                median0, median1, median2 = group0.median(), group1.median(), group2.median()
                
                all_values = pd.concat([group0, group1, group2])
                all_grades = [0]*len(group0) + [1]*len(group1) + [2]*len(group2)
                trend_corr, trend_p = stats.spearmanr(all_values, all_grades)
                
                univariate_results.append({
                    'Feature': col,
                    'G1_median': median0,
                    'G2_median': median1,
                    'G3/Cancer_median': median2,
                    'KW_Statistic': stat,
                    'P_value': p_value,
                    'Trend_Correlation': trend_corr,
                    'Trend_Direction': '↑' if trend_corr > 0 and p_value < 0.05 
                                     else '↓' if trend_corr < 0 and p_value < 0.05 else '—',
                    'Significance': '***' if p_value < 0.001 else '**' if p_value < 0.01 
                                  else '*' if p_value < 0.05 else ''
                })
        except Exception as e:
            continue
    
    univariate_df = pd.DataFrame(univariate_results).sort_values('P_value')
    
    print(f"✓ 成功分析 {len(univariate_df)} 个特征")
    
    univariate_df.to_csv(os.path.join(output_dir, '02_univariate_KW_analysis.csv'),
                        index=False, encoding='utf-8-sig')
    
    print("\n单因素Kruskal-Wallis检验结果（Top 20，按P值排序）：")
    display_cols = ['Feature', 'G1_median', 'G2_median', 'G3/Cancer_median',
                    'Trend_Direction', 'P_value', 'Significance']
    print(univariate_df[display_cols].head(20).to_string(index=False))
    
    significant_features = univariate_df[univariate_df['P_value'] < 0.05]['Feature'].tolist()
    print(f"\n✓ 单因素分析显著特征 (P<0.05): {len(significant_features)} 个")
    
    if len(significant_features) > 0:
        sig_df = univariate_df[univariate_df['P_value'] < 0.05][
            ['Feature', 'Trend_Direction', 'Trend_Correlation', 'P_value', 'Significance']]
        print(f"\n显著特征列表：")
        print(sig_df.to_string(index=False))
        
        sig_df.to_csv(os.path.join(output_dir, '02_significant_features.csv'),
                     index=False, encoding='utf-8-sig')
    
    plot_univariate_results(univariate_df, output_dir)
    
    return univariate_df, significant_features


def plot_univariate_results(univariate_df, output_dir):
    """可视化单因素分析结果 - 仅绘制关键图表"""
    
    # 曼哈顿图
    fig, ax = plt.subplots(figsize=(10, 8))
    
    top20 = univariate_df.head(20).copy()
    
    def get_color(p):
        if p < 0.001:
            return COLORS['danger']
        elif p < 0.01:
            return COLORS['warning']
        elif p < 0.05:
            return '#F4A261'
        else:
            return '#888888'
    
    colors = [get_color(p) for p in top20['P_value']]
    y_pos = range(len(top20))
    
    bars = ax.barh(y_pos, -np.log10(top20['P_value']),
                   color=colors, edgecolor='white', linewidth=1.5, alpha=0.85)
    
    ax.axvline(x=-np.log10(0.05), color=COLORS['danger'], linestyle='--',
               linewidth=2.5, label='P=0.05', alpha=0.7)
    ax.axvline(x=-np.log10(0.01), color=COLORS['warning'], linestyle=':',
               linewidth=2, label='P=0.01', alpha=0.6)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top20['Feature'], fontsize=10)
    ax.set_xlabel(r'$-log_{10}$(P-value)', fontsize=12, fontweight='bold')
    ax.set_title('Top 20 Features - Kruskal-Wallis Test',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_univariate_manhattan.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 保存图片: 02_univariate_manhattan.png")


def check_multicollinearity(X_encoded, significant_features, univariate_df, output_dir):
    """多重共线性检测 - 精简版"""
    print("\n" + "="*80)
    print("步骤6：多重共线性检测")
    print("="*80)
    
    if len(significant_features) <= 1:
        print("✓ 特征数量<=1，无需检测共线性")
        return significant_features
    
    X_significant = X_encoded[significant_features].copy().astype(float)
    
    print("\n计算特征间相关系数...")
    corr_matrix = X_significant.corr()
    
    # 层次聚类排序
    if len(significant_features) > 2:
        dissimilarity = 1 - corr_matrix.abs()
        Z = hierarchy.linkage(squareform(dissimilarity), method='average')
        dendro = hierarchy.dendrogram(Z, no_plot=True)
        reorder_idx = dendro['leaves']
        
        corr_matrix_clustered = corr_matrix.iloc[reorder_idx, reorder_idx]
        features_ordered = [significant_features[i] for i in reorder_idx]
    else:
        corr_matrix_clustered = corr_matrix
        features_ordered = significant_features
    
    # 绘制相关性热图
    fig, ax = plt.subplots(figsize=(12, 10))
    
    cmap = sns.diverging_palette(250, 10, s=80, l=55, n=200, as_cmap=True)
    mask = np.triu(np.ones_like(corr_matrix_clustered, dtype=bool), k=0)
    
    sns.heatmap(corr_matrix_clustered,
                mask=mask,
                cmap=cmap,
                center=0,
                vmin=-1, vmax=1,
                square=True,
                linewidths=2,
                linecolor='white',
                cbar_kws={'label': 'Correlation Coefficient', 'shrink': 0.8},
                annot=True,
                fmt='.2f',
                annot_kws={'size': 9},
                ax=ax)
    
    ax.set_xticklabels(features_ordered, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(features_ordered, rotation=0, fontsize=10)
    ax.set_title('Feature Correlation Matrix (Hierarchical Clustering)',
                 fontsize=13, fontweight='bold')
    
    # 标注高相关性
    for i in range(len(corr_matrix_clustered)):
        for j in range(i):
            val = corr_matrix_clustered.iloc[i, j]
            if abs(val) > 0.8:
                ax.plot(j + 0.5, i + 0.5, 'r*', markersize=15, zorder=2)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '03_correlation_heatmap.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 保存图片: 03_correlation_heatmap.png")
    
    # 检测高相关性特征对
    corr_abs = corr_matrix.abs()
    high_corr_pairs = []
    
    for i in range(len(corr_abs.columns)):
        for j in range(i+1, len(corr_abs.columns)):
            if corr_abs.iloc[i, j] > 0.8:
                high_corr_pairs.append({
                    'Feature1': corr_abs.columns[i],
                    'Feature2': corr_abs.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr_pairs:
        print(f"\n⚠ 发现{len(high_corr_pairs)}对高度相关的特征（|相关系数|>0.8）：")
        
        corr_pairs_df = pd.DataFrame(high_corr_pairs)
        print(corr_pairs_df.to_string(index=False))
        
        corr_pairs_df.to_csv(os.path.join(output_dir, '03_high_correlation_pairs.csv'),
                            index=False, encoding='utf-8-sig')
        
        # 处理共线性
        features_to_remove = set()
        for pair in high_corr_pairs:
            feat1 = pair['Feature1']
            feat2 = pair['Feature2']
            p1 = univariate_df[univariate_df['Feature'] == feat1]['P_value'].values[0]
            p2 = univariate_df[univariate_df['Feature'] == feat2]['P_value'].values[0]
            
            if p1 < p2:
                features_to_remove.add(feat2)
                print(f"  移除: {feat2} (P={p2:.4f}), 保留: {feat1} (P={p1:.4f})")
            else:
                features_to_remove.add(feat1)
                print(f"  移除: {feat1} (P={p1:.4f}), 保留: {feat2} (P={p2:.4f})")
        
        significant_features = [f for f in significant_features if f not in features_to_remove]
        print(f"\n✓ 移除共线性后保留 {len(significant_features)} 个特征")
    else:
        print("✓ 未发现严重的共线性问题（|相关系数|均<0.8）")
    
    return significant_features


def multivariate_analysis(X_encoded, y, significant_features, univariate_df, output_dir):
    """多因素有序Logistic回归分析"""
    print("\n" + "="*80)
    print("步骤7：多因素有序Logistic回归分析")
    print("="*80)
    
    if len(significant_features) == 0:
        print("⚠ 警告：无显著特征进行多因素分析")
        return None, []
    
    X_significant = X_encoded[significant_features].copy()
    y_clean = y.astype(int)
    
    n_samples = len(y_clean)
    n_features = len(significant_features)
    n_per_class = [sum(y_clean == i) for i in range(3)]
    min_class = min(n_per_class)
    samples_per_variable = min_class / n_features
    
    print(f"\n样本量检查：")
    print(f"  总样本数: {n_samples}")
    print(f"  特征数: {n_features}")
    print(f"  各类别样本数: G1={n_per_class[0]}, G2={n_per_class[1]}, G3/癌={n_per_class[2]}")
    
    if samples_per_variable < 10:
        print(f"⚠ 警告: 样本数/特征数 < 10")
        max_features = max(3, int(min_class / 10))
        print(f"推荐最多使用 {max_features} 个特征")
        
        if n_features > max_features:
            print(f"\n自动选择P值最小的 {max_features} 个特征...")
            top_features_df = univariate_df[univariate_df['Feature'].isin(significant_features)].head(max_features)
            significant_features = top_features_df['Feature'].tolist()
            X_significant = X_encoded[significant_features].copy()
            n_features = len(significant_features)
    
    X_significant = X_significant.astype(float)
    if X_significant.isnull().sum().sum() > 0 or np.isinf(X_significant.values).any():
        print("⚠ 数据清理中...")
        X_significant = X_significant.replace([np.inf, -np.inf], np.nan)
        valid_rows = X_significant.notna().all(axis=1)
        X_significant = X_significant[valid_rows]
        y_clean = y_clean[valid_rows]
    
    print(f"✓ 最终有效样本数: {len(y_clean)}")
    
    ordinal_success = False
    result = None
    
    try:
        print("\n拟合有序Logistic回归模型...")
        mod = OrderedModel(y_clean, X_significant, distr='logit')
        
        for method in ['bfgs', 'newton', 'nm']:
            try:
                print(f"  尝试优化方法: {method}...")
                result = mod.fit(method=method, disp=False, maxiter=1000)
                if result.mle_retvals['converged']:
                    print(f"  ✓ 使用 {method} 方法收敛成功")
                    ordinal_success = True
                    break
            except:
                continue
        
        if not ordinal_success:
            raise Exception("所有优化方法均失败")
        
    except:
        print(f"\n⚠ 有序Logistic回归失败")
        ordinal_success = False
    
    if ordinal_success and result is not None:
        print(f"\n✓ 模型拟合成功！")
        print(f"  AIC: {result.aic:.2f}")
        print(f"  BIC: {result.bic:.2f}")
        
        ordinal_results = []
        for feature in significant_features:
            try:
                if feature in result.params.index:
                    coef = result.params[feature]
                    pval = result.pvalues[feature]
                    stderr = result.bse[feature]
                    ci_lower, ci_upper = result.conf_int().loc[feature]
                    
                    or_value = np.exp(coef)
                    or_ci_lower = np.exp(ci_lower)
                    or_ci_upper = np.exp(ci_upper)
                    
                    ordinal_results.append({
                        'Feature': feature,
                        'Coefficient': coef,
                        'Std_Error': stderr,
                        'OR': or_value,
                        'OR_95%CI_Lower': or_ci_lower,
                        'OR_95%CI_Upper': or_ci_upper,
                        'P_value': pval,
                        'Significance': '***' if pval < 0.001 else '**' if pval < 0.01 
                                      else '*' if pval < 0.05 else ''
                    })
            except:
                continue
        
        if len(ordinal_results) > 0:
            multivariate_df = pd.DataFrame(ordinal_results).sort_values('P_value')
            
            print("\n有序Logistic回归结果（比例优势模型）")
            print("="*80)
            print(multivariate_df.to_string(index=False))
            
            multivariate_df.to_csv(os.path.join(output_dir, '04_ordinal_logistic_results.csv'),
                                  index=False, encoding='utf-8-sig')
            
            plot_multivariate_results(multivariate_df, output_dir)
            
            final_features = multivariate_df[multivariate_df['P_value'] < 0.05]['Feature'].tolist()
            return multivariate_df, final_features
    
    # 备选方案
    print("\n使用备选方案：多项Logistic回归...")
    
    lr_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                                 max_iter=1000, random_state=42, C=1.0)
    lr_multi.fit(X_significant, y_clean)
    
    print(f"✓ 多项模型训练完成")
    print(f"✓ 训练集准确率: {lr_multi.score(X_significant, y_clean):.4f}")
    
    multinomial_results = []
    for i, feature in enumerate(significant_features):
        coef_class0 = lr_multi.coef_[0][i]
        coef_class1 = lr_multi.coef_[1][i]
        coef_class2 = lr_multi.coef_[2][i]
        
        avg_coef = np.mean([coef_class0, coef_class1, coef_class2])
        max_abs_coef = np.abs([coef_class0, coef_class1, coef_class2]).max()
        
        multinomial_results.append({
            'Feature': feature,
            'Coef_G1': coef_class0,
            'Coef_G2': coef_class1,
            'Coef_G3/Cancer': coef_class2,
            'Importance': max_abs_coef
        })
    
    multivariate_df = pd.DataFrame(multinomial_results).sort_values('Importance', ascending=False)
    
    print("\n多项Logistic回归结果：")
    print(multivariate_df.to_string(index=False))
    
    multivariate_df.to_csv(os.path.join(output_dir, '04_multinomial_logistic_results.csv'),
                          index=False, encoding='utf-8-sig')
    
    plot_multivariate_results(multivariate_df, output_dir, method='multinomial')
    
    final_features = multivariate_df.nlargest(min(5, len(multivariate_df)), 'Importance')['Feature'].tolist()
    
    return multivariate_df, final_features


def plot_multivariate_results(multivariate_df, output_dir, method='ordinal'):
    """可视化多因素分析结果 - 仅绘制关键森林图"""
    
    if method == 'ordinal':
        # 森林图
        fig, ax = plt.subplots(figsize=(12, max(6, len(multivariate_df)*0.4)))
        
        df_sorted = multivariate_df.sort_values('OR', ascending=True)
        y_pos = range(len(df_sorted))
        
        for i, row in df_sorted.iterrows():
            color = COLORS['danger'] if row['P_value'] < 0.05 else 'gray'
            alpha = 0.9 if row['P_value'] < 0.05 else 0.4
            
            ax.plot([row['OR_95%CI_Lower'], row['OR_95%CI_Upper']], [i, i],
                    color=color, linewidth=3, alpha=alpha)
            ax.plot([row['OR_95%CI_Lower'], row['OR_95%CI_Upper']], [i, i],
                    '|', color=color, markersize=10, markeredgewidth=2, alpha=alpha)
            
            marker_size = 200 if row['P_value'] < 0.05 else 100
            ax.scatter(row['OR'], i, s=marker_size, c=color, edgecolor='white',
                      linewidth=2, alpha=alpha, marker='D')
        
        ax.axvline(x=1, color='black', linestyle='--', linewidth=2.5, alpha=0.7)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_sorted['Feature'], fontsize=10)
        ax.set_xlabel('Odds Ratio (OR) with 95% CI', fontsize=12, fontweight='bold')
        ax.set_title('Ordinal Logistic Regression - Forest Plot',
                     fontsize=13, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '04_forest_plot.png'),
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"📊 保存图片: 04_forest_plot.png")
        
    else:
        # 多项回归 - 系数热图
        fig, ax = plt.subplots(figsize=(8, max(6, len(multivariate_df)*0.4)))
        
        coef_data = multivariate_df[['Coef_G1', 'Coef_G2', 'Coef_G3/Cancer']].values
        
        cmap = sns.diverging_palette(250, 10, s=80, l=55, n=200, as_cmap=True)
        
        im = ax.imshow(coef_data, cmap=cmap, aspect='auto', vmin=-2, vmax=2)
        
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['G1', 'G2', 'G3/Cancer'], fontsize=11, fontweight='bold')
        ax.set_yticks(range(len(multivariate_df)))
        ax.set_yticklabels(multivariate_df['Feature'], fontsize=10)
        ax.set_title('Multinomial Logistic Regression - Coefficients',
                     fontsize=13, fontweight='bold')
        
        for i in range(len(multivariate_df)):
            for j in range(3):
                val = coef_data[i, j]
                text_color = 'white' if abs(val) > 1 else 'black'
                ax.text(j, i, f'{val:.2f}', ha="center", va="center",
                        color=text_color, fontsize=10, fontweight='bold')
        
        plt.colorbar(im, ax=ax, label='Coefficient Value')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '04_coef_heatmap.png'),
                   dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"📊 保存图片: 04_coef_heatmap.png")


def save_final_data(X_encoded, y, final_features, Surv, output_dir):
    """保存最终数据"""
    print("\n" + "="*80)
    print("步骤8：保存最终数据")
    print("="*80)
    
    if len(final_features) > 0:
        X_final = X_encoded[final_features].copy()
        X_final['Grade'] = y.values
        X_final = pd.concat([X_final, Surv], axis=1)
        X_final.to_csv(os.path.join(output_dir, '05_final_modeling_data.csv'),
                      encoding='utf-8-sig')
        print(f"✓ 最终建模特征数据: 05_final_modeling_data.csv ({len(final_features)} 个特征)")
    
    X_all = X_encoded.copy()
    X_all['Grade'] = y.values
    X_all = pd.concat([X_all, Surv], axis=1)
    X_all.to_csv(os.path.join(output_dir, '05_all_processed_data.csv'),
                encoding='utf-8-sig')
    print(f"✓ 所有处理后数据: 05_all_processed_data.csv ({X_encoded.shape[1]} 个特征)")


def generate_summary_report(output_dir, analysis_time, n_samples, n_features,
                           significant_features, final_features, univariate_df,
                           multivariate_df):
    """生成综合分析报告"""
    report_lines = []
    
    report_lines.append("="*80)
    report_lines.append("CLINICAL FEATURES ANALYSIS - SUMMARY REPORT")
    report_lines.append("="*80)
    report_lines.append(f"Analysis Time: {analysis_time}")
    report_lines.append("="*80)
    report_lines.append("")
    
    report_lines.append("1. DATA OVERVIEW")
    report_lines.append("-"*80)
    report_lines.append(f"Total Samples: {n_samples}")
    report_lines.append(f"Total Features: {n_features}")
    report_lines.append("")
    
    report_lines.append("2. UNIVARIATE ANALYSIS (Kruskal-Wallis Test)")
    report_lines.append("-"*80)
    report_lines.append(f"Significant Features (P<0.05): {len(significant_features)}")
    report_lines.append(f"Features: {', '.join(significant_features[:20])}")
    report_lines.append("")
    
    if len(univariate_df) > 0:
        report_lines.append("Top 10 Most Significant Features:")
        top10 = univariate_df.head(10)
        for _, row in top10.iterrows():
            report_lines.append(f"  {row['Feature']}: P={row['P_value']:.4f}, Trend={row['Trend_Direction']}")
    report_lines.append("")
    
    report_lines.append("3. MULTIVARIATE ANALYSIS")
    report_lines.append("-"*80)
    report_lines.append(f"Final Modeling Features: {len(final_features)}")
    report_lines.append(f"Features: {', '.join(final_features)}")
    report_lines.append("")
    
    if multivariate_df is not None and len(multivariate_df) > 0:
        report_lines.append("Significant Results:")
        if 'OR' in multivariate_df.columns:
            for _, row in multivariate_df[multivariate_df['P_value'] < 0.05].iterrows():
                report_lines.append(f"  {row['Feature']}: OR={row['OR']:.3f}, "
                                  f"P={row['P_value']:.4f}")
    
    report_lines.append("")
    report_lines.append("="*80)
    
    report_path = os.path.join(output_dir, '00_SUMMARY_REPORT.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    
    print(f"\n💾 综合报告已保存: 00_SUMMARY_REPORT.txt")


def main():
    """主函数"""
    analysis_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    print("="*80)
    print("CLINICAL FEATURES ANALYSIS - KRUSKAL-WALLIS + ORDINAL LOGISTIC")
    print(f"Analysis Time: {analysis_time}")
    print("="*80)
    
    output_dir = create_output_folder('./clinical')
    
    X, y, features, Surv = load_and_encode_data()
    n_samples_original = len(y)
    n_features_original = len(features)
    
    X, cols_dropped = analyze_missing_values(X, output_dir)
    
    X, categorical_vars, continuous_vars = impute_missing_values(X)
    
    X_encoded = encode_features(X, categorical_vars)
    
    univariate_df, significant_features = univariate_analysis(X_encoded, y, output_dir)
    
    if len(significant_features) > 0:
        significant_features = check_multicollinearity(
            X_encoded, significant_features, univariate_df, output_dir)
    
    multivariate_df, final_features = multivariate_analysis(
        X_encoded, y, significant_features, univariate_df, output_dir)
    
    if final_features is None or len(final_features) == 0:
        print("\n⚠ 多因素分析未发现显著特征，使用单因素显著特征")
        final_features = significant_features
    
    save_final_data(X_encoded, y, final_features, Surv, output_dir)
    
    generate_summary_report(
        output_dir, analysis_time, n_samples_original, n_features_original,
        significant_features, final_features, univariate_df, multivariate_df)
    
    print("\n" + "="*80)
    print("✅ 分析完成！所有结果已保存到 ./clinical/ 文件夹")
    print("="*80)


if __name__ == '__main__':
    main()

CLINICAL FEATURES ANALYSIS - KRUSKAL-WALLIS + ORDINAL LOGISTIC
Analysis Time: 2026-04-23 19:19:26
步骤1：数据加载与标签编码
✓ 原始样本数: 546
✓ 标签有效样本数: 546

标签分布: G1=226, G2=228, G3/癌=92
总样本数: 546

步骤2：缺失值分析与处理

各特征缺失率：
Feature  Missing_Rate(%)
 CA72-4        81.318681
  CA125        72.527473
     TG        64.285714
    AFP        61.721612
    CEA        39.560440
 CA19-9        38.827839
    LDH        19.413919
     TC        18.681319
   eGFR         9.157509
    FBG         7.509158
    AST         7.509158
    ALT         3.663004
    BUN         3.113553
    SCr         3.113553
    ALB         2.747253
    PLR         2.380952
    NLR         2.197802
    SII         1.831502
📊 保存图片: 01_missing_rate_barplot.png

⚠ 剔除缺失率≥25%的特征 (6个): ['CA19-9', 'CA125', 'CA72-4', 'CEA', 'AFP', 'TG']

步骤3：缺失值插补
使用MICE多重插补法处理连续变量缺失值...
✓ 连续变量插补完成

步骤4：特征编码
✓ 编码前特征数: 25
✓ 编码后特征数: 25

步骤5：单因素分析（Kruskal-Wallis检验）
✓ 成功分析 25 个特征

单因素Kruskal-Wallis检验结果（Top 20，按P值排序）：
Feature   G1_median   G2_median  G3/Cancer_median 

In [4]:
"""
================================================================================
                    临床特征有效性验证系统
================================================================================

功能：
    读取步骤4-5的特征筛选结果，对其进行多维度验证
    ✓ 验证特征在各分级中的分布差异
    ✓ 验证特征与患者生存的相关性

输入：
    • ./clinical/04_ordinal_logistic_results.csv (多因素分析结果)
    • ./clinical/05_final_modeling_data.csv (处理后的数据)

输出：
    • PDF矢量图(专业出版级)
    • CSV统计表格

特点：
    • 所有图表输出为PDF格式
    • P值显示在图表下方(无背景)
    • 自动读取前一步筛选的特征

================================================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
import os
import re
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ==================== 配置 ====================
COLORS = {
    'primary': '#2E86AB',
    'success': '#06A77D',
    'warning': '#F77F00',
    'danger': '#D62828',
    'info': '#4EA8DE',
    'accent': '#A23B72'
}

# PDF输出设置
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42                # PDF嵌入字体
plt.rcParams['ps.fonttype'] = 42                 # 确保字体正确显示


# ==================== 工具函数 ====================

def sanitize_filename(filename):
    """清理文件名中的非法字符"""
    return re.sub(r'[<>:"/\\|?*]', '_', filename)


def detect_variable_type(data, feature, max_unique_count=10):
    """
    自动检测变量类型
    
    Parameters
    ----------
    data : array-like
        特征数据
    max_unique_count : int
        唯一值数量阈值(≤此值判为分类变量)
    
    Returns
    -------
    var_type : str
        'continuous' 或 'categorical'
    info : dict
        唯一值数、是否二分类等信息
    """
    clean_data = data[~np.isnan(data)]
    n_unique = len(np.unique(clean_data))
    
    is_categorical = n_unique <= max_unique_count
    var_type = 'categorical' if is_categorical else 'continuous'
    
    return var_type, {
        'n_unique': n_unique,
        'is_binary': n_unique == 2,
        'unique_values': np.unique(clean_data) if is_categorical else None
    }


def remove_outliers_iqr(data, multiplier=3.0):
    """
    IQR方法检测异常值
    
    公式：
        下界 = Q1 - multiplier × IQR
        上界 = Q3 + multiplier × IQR
    """
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    
    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr
    
    mask = (data >= lower_bound) & (data <= upper_bound)
    n_outliers = (~mask).sum()
    
    return mask, (lower_bound, upper_bound), {
        'n_outliers': n_outliers,
        'outlier_ratio': n_outliers / len(data) * 100 if len(data) > 0 else 0,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }


def format_pvalue(p_value):
    """
    格式化P值显示
    
    Returns
    -------
    formatted : str
        'P<0.001' 或 'P=xxx'
    sig : str
        '***' 或 '**' 或 '*' 或 'ns'
    """
    if p_value < 0.001:
        return 'P < 0.001', '***'
    elif p_value < 0.01:
        return f'P = {p_value:.4f}', '**'
    elif p_value < 0.05:
        return f'P = {p_value:.4f}', '*'
    else:
        return f'P = {p_value:.4f}', 'ns'


def save_pdf_figure(fig, output_dir, filename_base):
    """
    保存PDF矢量图
    
    Parameters
    ----------
    fig : matplotlib.figure.Figure
        图表对象
    output_dir : str
        输出目录
    filename_base : str
        文件名(不含扩展名)
    """
    filepath = os.path.join(output_dir, f'{filename_base}.pdf')
    fig.savefig(filepath, format='pdf', bbox_inches='tight', facecolor='white')
    
    # 同时保存PNG作为快速预览
    filepath_png = os.path.join(output_dir, f'{filename_base}.png')
    fig.savefig(filepath_png, format='png', dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"  💾 保存: {filename_base}.pdf")


def load_multivariate_features(input_dir='./clinical'):
    """
    从步骤4的多因素分析结果中读取筛选的特征
    
    Returns
    -------
    features : list
        多因素显著特征列表
    feature_data : dict
        特征的统计信息(P值、OR等)
    """
    result_path = os.path.join(input_dir, '04_ordinal_logistic_results.csv')
    
    if not os.path.exists(result_path):
        print(f"✗ 错误: 找不到文件 {result_path}")
        print("  请先运行步骤4(多因素分析)")
        return [], {}
    
    try:
        df = pd.read_csv(result_path)
        
        # 筛选P<0.05的特征
        features = df[df['P_value'] < 0.05]['Feature'].tolist()
        
        # 保存特征数据
        feature_data = dict(zip(df['Feature'], df['P_value']))
        
        print(f"✓ 从步骤4加载{len(features)}个显著特征")
        print(f"  特征列表: {', '.join(features)}")
        
        return features, feature_data
    except Exception as e:
        print(f"✗ 读取失败: {e}")
        return [], {}


def load_processed_data(input_dir='./clinical'):
    """
    从步骤5读取处理后的数据
    
    Returns
    -------
    data : pd.DataFrame
    """
    data_path = os.path.join(input_dir, '05_final_modeling_data.csv')
    
    if not os.path.exists(data_path):
        print(f"✗ 错误: 找不到文件 {data_path}")
        return None
    
    try:
        data = pd.read_csv(data_path, index_col=0)
        print(f"✓ 加载数据: {data.shape[0]}个样本, {data.shape[1]}个特征")
        return data
    except Exception as e:
        print(f"✗ 读取失败: {e}")
        return None


# ==================== 主验证函数 ====================

def validate_feature_effectiveness(input_dir='./clinical',
                                   output_dir='./clinical',
                                   survival_time_col='time',
                                   survival_event_col='event',
                                   label_col='Grade'):
    """
    特征有效性验证主函数(读取前一步结果)
    
    所有输出图表为PDF格式
    """
    print("\n" + "="*80)
    print("特征有效性验证 (读取步骤4-5的结果)")
    print("所有图表输出为PDF格式")
    print("="*80)
    
    # 步骤1: 加载前一步结果
    features, feature_data = load_multivariate_features(input_dir)
    if len(features) == 0:
        print("\n✗ 错误: 无法加载特征列表")
        return
    
    data = load_processed_data(input_dir)
    if data is None:
        print("\n✗ 错误: 无法加载数据")
        return
    
    # 步骤2: 创建输出目录
    validation_dir = os.path.join(output_dir, '06_feature_validation')
    os.makedirs(validation_dir, exist_ok=True)
    
    # 步骤3: 执行验证
    print(f"\n验证{len(features)}个特征...")
    print("-" * 80)
    
    # 分级分布验证
    validate_distribution_by_grade(data, features, label_col, validation_dir)
    
    # 生存分析验证
    if survival_time_col in data.columns and survival_event_col in data.columns:
        validate_survival_analysis(data, features, survival_time_col,
                                  survival_event_col, validation_dir)
    else:
        print(f"⚠ 警告: 缺少生存数据列({survival_time_col}, {survival_event_col})")
    
    print("\n" + "="*80)
    print("✅ 特征有效性验证完成")
    print(f"📁 结果保存到: {validation_dir}")
    print(f"📊 输出格式: PDF(编辑) + PNG(预览)")
    print("="*80)


# ==================== 分级分布验证 ====================

def validate_distribution_by_grade(data, features, label_col, output_dir):
    """
    验证特征在各分级中的分布是否存在显著差异
    """
    print(f"\n📊 分级分布验证 ({len(features)}个特征)")
    print("-" * 80)
    
    test_results = []
    
    for feature in features:
        print(f"\n特征: {feature}")
        
        # 提取数据并清理缺失值
        feature_data = data[[feature, label_col]].dropna()
        
        if len(feature_data) == 0:
            print(f"  ⚠ 跳过: 无有效数据")
            continue
        
        # 检测变量类型
        var_type, var_info = detect_variable_type(feature_data[feature].values, feature)
        print(f"  变量类型: {var_type} (唯一值数: {var_info['n_unique']})")
        
        # 按分级分组
        unique_grades = sorted(feature_data[label_col].unique())
        grade_groups = [feature_data[feature_data[label_col] == g][feature].values
                       for g in unique_grades]
        
        safe_name = sanitize_filename(feature)
        
        # 根据变量类型选择统计方法
        if var_type == 'categorical':
            # 分类变量：卡方检验或Fisher精确检验
            contingency_table = pd.crosstab(feature_data[feature], feature_data[label_col])
            
            if contingency_table.size < 5 or contingency_table.min().min() < 5:
                if var_info['is_binary'] and len(unique_grades) == 2:
                    _, p_value = stats.fisher_exact(contingency_table)
                    test_name = "Fisher"
                else:
                    chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
                    test_name = "χ²"
            else:
                chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
                test_name = "χ²"
            
            p_str, sig = format_pvalue(p_value)
            print(f"  {test_name} 检验: {p_str} {sig}")
            plot_categorical_distribution(feature_data, feature, label_col,
                                         unique_grades, p_value, test_name,
                                         safe_name, output_dir)
        else:
            # 连续变量：Kruskal-Wallis检验
            h_stat, p_value = stats.kruskal(*grade_groups)
            test_name = "Kruskal-Wallis"
            
            p_str, sig = format_pvalue(p_value)
            print(f"  {test_name} 检验: {p_str} {sig}")
            plot_continuous_distribution(feature_data, feature, label_col,
                                        unique_grades, grade_groups, p_value,
                                        safe_name, output_dir)
        
        # 记录结果
        test_results.append({
            'Feature': feature,
            'Variable_Type': var_type,
            'Test_Method': test_name,
            'P_value': p_value,
            'P_string': p_str,
            'Significance': sig
        })
    
    # 保存结果表格
    if len(test_results) > 0:
        test_df = pd.DataFrame(test_results)
        test_df.to_csv(os.path.join(output_dir, 'grade_distribution_statistics.csv'),
                      index=False, encoding='utf-8-sig')
        print(f"\n✓ 统计结果保存: grade_distribution_statistics.csv")
        
        # 绘制汇总图
        plot_distribution_summary(test_df, output_dir)


def plot_categorical_distribution(feature_data, feature, label_col,
                                  unique_grades, p_value, test_name,
                                  safe_feature_name, output_dir):
    """
    绘制分类变量的分布图(堆叠百分比 + 分组计数)
    PDF输出
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('white')
    
    grade_labels_map = {0: 'G1', 1: 'G2', 2: 'G3/Cancer'}
    grade_labels = [grade_labels_map.get(g, f'Grade{g}') for g in unique_grades]
    colors = [COLORS['success'], COLORS['warning'], COLORS['danger']]
    
    # 左图：堆叠百分比
    ax1 = axes[0]
    contingency_pct = pd.crosstab(feature_data[label_col], feature_data[feature],
                                  normalize='index') * 100
    contingency_pct.index = grade_labels
    contingency_pct.plot(kind='bar', stacked=True, ax=ax1,
                        color=colors[:len(contingency_pct.columns)],
                        edgecolor='white', linewidth=2, alpha=0.85)
    
    ax1.set_xlabel('Grade', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Percentage (%)', fontsize=11, fontweight='bold')
    ax1.set_title(f'{feature} - Stacked Distribution', fontsize=12, fontweight='bold')
    ax1.legend(title=feature, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
    ax1.grid(axis='y', alpha=0.3)
    
    # 显著性标注(下方，无背景)
    p_str, sig = format_pvalue(p_value)
    ax1.text(0.5, -0.25, f'{test_name} {p_str} {sig}',
            transform=ax1.transAxes, ha='center', fontsize=10, fontweight='bold')
    
    # 右图：分组计数
    ax2 = axes[1]
    contingency_count = pd.crosstab(feature_data[label_col], feature_data[feature])
    contingency_count.index = grade_labels
    contingency_count.plot(kind='bar', ax=ax2, color=colors[:len(contingency_count.columns)],
                          edgecolor='white', linewidth=2, alpha=0.85)
    
    ax2.set_xlabel('Grade', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
    ax2.set_title(f'{feature} - Count by Grade', fontsize=12, fontweight='bold')
    ax2.legend(title=feature, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
    ax2.grid(axis='y', alpha=0.3)
    
    # 显著性标注(下方，无背景)
    ax2.text(0.5, -0.25, f'{test_name} {p_str} {sig}',
            transform=ax2.transAxes, ha='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, f'{safe_feature_name}_distribution')
    plt.close()


def plot_continuous_distribution(feature_data, feature, label_col,
                                 unique_grades, grade_groups, p_value,
                                 safe_feature_name, output_dir):
    """
    绘制连续变量的分布图(小提琴图 + 箱线图)
    PDF输出
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('white')
    
    # 异常值检测和移除
    all_data = feature_data[feature].values
    mask_clean, (lower, upper), outlier_info = remove_outliers_iqr(all_data)
    
    if outlier_info['n_outliers'] > 0:
        print(f"  检测到{outlier_info['n_outliers']}个异常值(已移除)")
    
    feature_data_clean = feature_data[mask_clean].copy()
    grade_groups_clean = [feature_data_clean[feature_data_clean[label_col] == g][feature].values
                         for g in unique_grades]
    
    grade_labels_map = {0: 'G1', 1: 'G2', 2: 'G3/Cancer'}
    grade_labels = [grade_labels_map.get(g, f'Grade{g}') for g in unique_grades]
    colors = [COLORS['success'], COLORS['warning'], COLORS['danger']]
    
    # 左图：小提琴图
    ax1 = axes[0]
    parts = ax1.violinplot(grade_groups_clean, positions=range(len(grade_groups_clean)),
                           showmeans=True, showmedians=True, widths=0.7)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i % len(colors)])
        pc.set_alpha(0.7)
    
    ax1.set_xticks(range(len(grade_groups_clean)))
    ax1.set_xticklabels(grade_labels)
    ax1.set_ylabel(f'{feature} Value', fontsize=11, fontweight='bold')
    ax1.set_title(f'{feature} - Violin Distribution', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # 显著性标注(下方，无背景)
    p_str, sig = format_pvalue(p_value)
    ax1.text(0.5, -0.15, f'Kruskal-Wallis {p_str} {sig}',
            transform=ax1.transAxes, ha='center', fontsize=10, fontweight='bold')
    
    # 右图：箱线图
    ax2 = axes[1]
    bp = ax2.boxplot(grade_groups_clean, positions=range(len(grade_groups_clean)),
                     widths=0.6, patch_artist=True, showfliers=False,
                     boxprops=dict(linewidth=1.5),
                     medianprops=dict(color='red', linewidth=2))
    
    for patch, color in zip(bp['boxes'], colors[:len(grade_groups_clean)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax2.set_xticks(range(len(grade_groups_clean)))
    ax2.set_xticklabels(grade_labels)
    ax2.set_ylabel(f'{feature} Value', fontsize=11, fontweight='bold')
    ax2.set_title(f'{feature} - Boxplot Comparison', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    # 显著性标注(下方，无背景)
    ax2.text(0.5, -0.15, f'Kruskal-Wallis {p_str} {sig}',
            transform=ax2.transAxes, ha='center', fontsize=10, fontweight='bold')
    
    # 成对比较(Post-hoc)
    if p_value < 0.05 and len(grade_groups_clean) > 1:
        pairs = list(combinations(range(len(grade_groups_clean)), 2))
        y_max = max([max(g) for g in grade_groups_clean if len(g) > 0])
        y_start = y_max * 1.05
        y_step = y_max * 0.05
        
        for idx, (i, j) in enumerate(pairs[:3]):
            if len(grade_groups_clean[i]) > 0 and len(grade_groups_clean[j]) > 0:
                _, pair_p = stats.mannwhitneyu(grade_groups_clean[i], grade_groups_clean[j])
                if pair_p < 0.05:
                    y_pos = y_start + idx * y_step
                    ax2.plot([i, i, j, j], [y_pos, y_pos + y_step*0.2, y_pos + y_step*0.2, y_pos],
                            'k-', linewidth=1)
                    sig_pair = '*' if pair_p < 0.05 else 'ns'
                    ax2.text((i+j)/2, y_pos + y_step*0.3, sig_pair, ha='center', fontsize=9)
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, f'{safe_feature_name}_distribution')
    plt.close()


def plot_distribution_summary(test_df, output_dir):
    """
    绘制分级分布验证的总结图(P值条形图)
    PDF输出
    """
    fig, ax = plt.subplots(figsize=(10, max(6, len(test_df)*0.3)))
    fig.patch.set_facecolor('white')
    
    # P值条形图
    colors = [COLORS['danger'] if p < 0.001 else COLORS['warning'] if p < 0.01
             else COLORS['accent'] if p < 0.05 else '#888888'
             for p in test_df['P_value']]
    
    bars = ax.barh(range(len(test_df)), -np.log10(test_df['P_value']),
                   color=colors, edgecolor='white', linewidth=1.5, alpha=0.85)
    
    ax.axvline(x=-np.log10(0.05), color=COLORS['danger'], linestyle='--',
              linewidth=2, label='P=0.05', alpha=0.7)
    
    labels = [f"{row['Feature']} ({row['Variable_Type'][0].upper()})"
             for _, row in test_df.iterrows()]
    ax.set_yticks(range(len(test_df)))
    ax.set_yticklabels(labels, fontsize=9)
    
    ax.set_xlabel(r'$-log_{10}$(P-value)', fontsize=11, fontweight='bold')
    ax.set_title('Grade Distribution Test Results', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, 'grade_distribution_summary')
    plt.close()


# ==================== 生存分析验证 ====================

def validate_survival_analysis(data, features, time_col, event_col, output_dir):
    """
    验证特征与生存的相关性
    """
    print(f"\n📈 生存分析验证 ({len(features)}个特征)")
    print("-" * 80)
    
    survival_results = []
    
    for feature in features:
        print(f"\n特征: {feature}")
        
        # 提取生存数据
        survival_data = data[[feature, time_col, event_col]].dropna()
        
        if len(survival_data) == 0:
            print(f"  ⚠ 跳过: 无有效数据")
            continue
        
        # 检测变量类型
        var_type, var_info = detect_variable_type(survival_data[feature].values, feature)
        print(f"  变量类型: {var_type}")
        
        safe_name = sanitize_filename(feature)
        
        if var_type == 'categorical':
            # 分类变量：按类别分组
            plot_categorical_survival(survival_data, feature, time_col, event_col,
                                     var_info, safe_name, output_dir, survival_results)
        else:
            # 连续变量：中位值和三分位值分割
            plot_continuous_survival(survival_data, feature, time_col, event_col,
                                    safe_name, output_dir, survival_results)
    
    # 保存结果表格
    if len(survival_results) > 0:
        survival_df = pd.DataFrame(survival_results)
        survival_df.to_csv(os.path.join(output_dir, 'survival_analysis_statistics.csv'),
                          index=False, encoding='utf-8-sig')
        print(f"\n✓ 统计结果保存: survival_analysis_statistics.csv")
        
        # 绘制汇总图
        plot_survival_summary(survival_df, output_dir)


def plot_categorical_survival(survival_data, feature, time_col, event_col,
                              var_info, safe_feature_name, output_dir, survival_results):
    """
    分类变量生存曲线(Kaplan-Meier)
    PDF输出
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor('white')
    
    kmf = KaplanMeierFitter()
    colors = [COLORS['info'], COLORS['danger']]
    unique_values = var_info['unique_values']
    
    # 绘制各类别的KM曲线
    for idx, value in enumerate(unique_values):
        mask = survival_data[feature] == value
        n_subjects = mask.sum()
        
        kmf.fit(survival_data[mask][time_col],
               survival_data[mask][event_col],
               label=f'{feature}={value} (n={n_subjects})')
        kmf.plot_survival_function(ax=ax, ci_show=True, color=colors[idx % len(colors)],
                                  linewidth=2.5, alpha=0.8)
    
    # Log-rank检验
    if len(unique_values) == 2:
        results = logrank_test(
            survival_data[survival_data[feature] == unique_values[0]][time_col],
            survival_data[survival_data[feature] == unique_values[1]][time_col],
            survival_data[survival_data[feature] == unique_values[0]][event_col],
            survival_data[survival_data[feature] == unique_values[1]][event_col]
        )
    else:
        results = multivariate_logrank_test(survival_data[time_col],
                                           survival_data[feature],
                                           survival_data[event_col])
    p_value = results.p_value
    p_str, sig = format_pvalue(p_value)
    
    print(f"  Log-rank {p_str} {sig}")
    
    ax.set_xlabel('Time (months)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Survival Probability', fontsize=11, fontweight='bold')
    ax.set_title(f'{feature} - Kaplan-Meier Survival Curve', fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # 显著性标注(下方，无背景)
    ax.text(0.02, 0.05, f'Log-rank {p_str} {sig}',
           transform=ax.transAxes, ha='left', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, f'{safe_feature_name}_survival')
    plt.close()
    
    survival_results.append({
        'Feature': feature,
        'Type': 'Categorical',
        'Cutoff': f'{unique_values[0]} vs {unique_values[1]}',
        'P_value': p_value,
        'P_string': p_str,
        'Significance': sig
    })


def plot_continuous_survival(survival_data, feature, time_col, event_col,
                             safe_feature_name, output_dir, survival_results):
    """
    连续变量生存曲线(中位值分割 + 三分位值分割)
    PDF输出
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.patch.set_facecolor('white')
    
    # 中位值分割
    median = survival_data[feature].median()
    survival_data['group_median'] = (survival_data[feature] > median).astype(int)
    
    kmf = KaplanMeierFitter()
    for group_id, label, color in [(0, f'≤{median:.2f}', COLORS['info']),
                                     (1, f'>{median:.2f}', COLORS['danger'])]:
        mask = survival_data['group_median'] == group_id
        kmf.fit(survival_data[mask][time_col], survival_data[mask][event_col],
               label=f'{label} (n={mask.sum()})')
        kmf.plot_survival_function(ax=axes[0], ci_show=True, color=color,
                                  linewidth=2.5, alpha=0.8)
    
    results_median = logrank_test(
        survival_data[survival_data['group_median'] == 0][time_col],
        survival_data[survival_data['group_median'] == 1][time_col],
        survival_data[survival_data['group_median'] == 0][event_col],
        survival_data[survival_data['group_median'] == 1][event_col]
    )
    p_median = results_median.p_value
    p_str_median, sig_median = format_pvalue(p_median)
    
    print(f"  中位值分割: {p_str_median} {sig_median}")
    
    axes[0].set_xlabel('Time (months)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Survival Probability', fontsize=11, fontweight='bold')
    axes[0].set_title(f'{feature} - Median Split', fontsize=12, fontweight='bold')
    axes[0].legend(loc='best', fontsize=10)
    axes[0].grid(alpha=0.3)
    
    # 显著性标注(下方，无背景)
    axes[0].text(0.02, 0.05, f'Log-rank {p_str_median} {sig_median}',
                transform=axes[0].transAxes, ha='left', fontsize=10, fontweight='bold')
    
    # 三分位值分割
    q1 = survival_data[feature].quantile(0.25)
    q3 = survival_data[feature].quantile(0.75)
    survival_data['group_tertile'] = pd.cut(survival_data[feature],
                                            bins=[-np.inf, q1, q3, np.inf],
                                            labels=['Low', 'Med', 'High'])
    
    colors_tertile = [COLORS['success'], COLORS['warning'], COLORS['danger']]
    for group_name, color in zip(['Low', 'Med', 'High'], colors_tertile):
        mask = survival_data['group_tertile'] == group_name
        if mask.sum() > 0:
            kmf.fit(survival_data[mask][time_col], survival_data[mask][event_col],
                   label=f'{group_name} (n={mask.sum()})')
            kmf.plot_survival_function(ax=axes[1], ci_show=True, color=color,
                                      linewidth=2.5, alpha=0.8)
    
    results_tertile = multivariate_logrank_test(survival_data[time_col],
                                               survival_data['group_tertile'],
                                               survival_data[event_col])
    p_tertile = results_tertile.p_value
    p_str_tertile, sig_tertile = format_pvalue(p_tertile)
    
    print(f"  三分位分割: {p_str_tertile} {sig_tertile}")
    
    axes[1].set_xlabel('Time (months)', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Survival Probability', fontsize=11, fontweight='bold')
    axes[1].set_title(f'{feature} - Tertile Split', fontsize=12, fontweight='bold')
    axes[1].legend(loc='best', fontsize=10)
    axes[1].grid(alpha=0.3)
    
    # 显著性标注(下方，无背景)
    axes[1].text(0.02, 0.05, f'Log-rank {p_str_tertile} {sig_tertile}',
                transform=axes[1].transAxes, ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, f'{safe_feature_name}_survival')
    plt.close()
    
    # 记录结果
    survival_results.append({
        'Feature': feature,
        'Type': 'Continuous',
        'Cutoff': f'Median={median:.3f}',
        'P_value': p_median,
        'P_string': p_str_median,
        'Significance': sig_median
    })
    
    survival_results.append({
        'Feature': feature,
        'Type': 'Continuous',
        'Cutoff': f'Q1={q1:.3f}, Q3={q3:.3f}',
        'P_value': p_tertile,
        'P_string': p_str_tertile,
        'Significance': sig_tertile
    })


def plot_survival_summary(survival_df, output_dir):
    """
    生存分析汇总图(P值条形图 + 结果表格)
    PDF输出
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    fig.patch.set_facecolor('white')
    
    # 上图：P值条形图
    ax1 = axes[0]
    
    labels = [f"{row['Feature']} - {row['Cutoff'][:20]}"
             for _, row in survival_df.iterrows()]
    colors = [COLORS['danger'] if p < 0.001 else COLORS['warning'] if p < 0.01
             else COLORS['accent'] if p < 0.05 else '#888888'
             for p in survival_df['P_value']]
    
    bars = ax1.barh(range(len(survival_df)), -np.log10(survival_df['P_value']),
                    color=colors, edgecolor='white', linewidth=1.5, alpha=0.85)
    
    ax1.axvline(x=-np.log10(0.05), color=COLORS['danger'], linestyle='--',
               linewidth=2, label='P=0.05', alpha=0.7)
    
    ax1.set_yticks(range(len(survival_df)))
    ax1.set_yticklabels(labels, fontsize=9)
    ax1.set_xlabel(r'$-log_{10}$(P-value)', fontsize=11, fontweight='bold')
    ax1.set_title('Survival Analysis Results', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(axis='x', alpha=0.3)
    ax1.invert_yaxis()
    
    # 下图：结果表格
    ax2 = axes[1]
    ax2.axis('off')
    
    table_data = []
    for _, row in survival_df.iterrows():
        table_data.append([
            row['Feature'],
            row['Type'][:4],
            row['Cutoff'][:25],
            row['P_string'],
            row['Significance']
        ])
    
    table = ax2.table(cellText=table_data,
                     colLabels=['Feature', 'Type', 'Cutoff', 'P-value', 'Sig'],
                     cellLoc='center', loc='center',
                     colWidths=[0.2, 0.1, 0.35, 0.2, 0.1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # 表头样式
    for i in range(5):
        table[(0, i)].set_facecolor(COLORS['primary'])
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # 单元格样式
    for i in range(1, len(table_data) + 1):
        for j in range(5):
            cell = table[(i, j)]
            cell.set_edgecolor('#CCC')
            if i % 2 == 0:
                cell.set_facecolor('#F9F9F9')
    
    plt.tight_layout()
    
    # 保存PDF
    save_pdf_figure(fig, output_dir, 'survival_analysis_summary')
    plt.close()


# ==================== 主程序 ====================

if __name__ == '__main__':
    """
    使用方法：
    
    1. 确保已完成步骤4-5，生成：
       - ./clinical/04_ordinal_logistic_results.csv
       - ./clinical/05_final_modeling_data.csv
    
    2. 运行本脚本：
       python step6_feature_validation.py
    
    3. 结果保存在：
       ./clinical/06_feature_validation/
       所有图表输出为PDF格式
    """
    
    print("\n" + "="*80)
    print("临床特征有效性验证 (PDF版)")
    print("输出格式: PDF(编辑) + PNG(预览)")
    print("="*80)
    
    validate_feature_effectiveness(
        input_dir='./clinical',
        output_dir='./clinical',
        survival_time_col='time',           # 列名: time
        survival_event_col='event',         # 列名: event
        label_col='Grade'                   # ← 改为你的列名(如需)
    )


临床特征有效性验证 (PDF版)
输出格式: PDF(编辑) + PNG(预览)

特征有效性验证 (读取步骤4-5的结果)
所有图表输出为PDF格式
✓ 从步骤4加载3个显著特征
  特征列表: size, RIF%, RSF%
✓ 加载数据: 546个样本, 6个特征

验证3个特征...
--------------------------------------------------------------------------------

📊 分级分布验证 (3个特征)
--------------------------------------------------------------------------------

特征: size
  变量类型: continuous (唯一值数: 546)
  Kruskal-Wallis 检验: P < 0.001 ***
  检测到5个异常值(已移除)
  💾 保存: size_distribution.pdf

特征: RIF%
  变量类型: continuous (唯一值数: 545)
  Kruskal-Wallis 检验: P = 0.0011 **
  检测到9个异常值(已移除)
  💾 保存: RIF%_distribution.pdf

特征: RSF%
  变量类型: continuous (唯一值数: 545)
  Kruskal-Wallis 检验: P = 0.0075 **
  💾 保存: RSF%_distribution.pdf

✓ 统计结果保存: grade_distribution_statistics.csv
  💾 保存: grade_distribution_summary.pdf

📈 生存分析验证 (3个特征)
--------------------------------------------------------------------------------

特征: size
  变量类型: continuous
  中位值分割: P < 0.001 ***
  三分位分割: P = 0.0013 **
  💾 保存: size_survival.pdf

特征: RIF%
  变量类型: continuous
  中位值分割: P